# Pairwise Cohen's Kappa Agreement Analysis

This notebook calculates pairwise Cohen's Kappa agreement scores for the **conspiracy classification** task across all pairs of human annotators.

## Analysis:
- Identifies all annotator pairs who annotated the same texts
- Calculates Cohen's Kappa for each pair on conspiracy classification (yes/no)
- Generates pairwise agreement matrix and summary statistics
- Exports results for paper inclusion

## Output:
- Pairwise Kappa matrix (table format)
- Summary statistics (mean, min, max Kappa scores)
- CSV export for paper inclusion
- LaTeX table format for paper inclusion


In [1]:
# Setup & Imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set paths
BASE = Path('/home/ashrafs/projects/Conspiracy-Proj')
DATA = BASE / 'Data'
PUB_TABLES = BASE / 'publication' / 'tables'
PUB_TABLES.mkdir(exist_ok=True, parents=True)

print("✓ Setup complete!")
print(f"Data directory: {DATA}")
print(f"Tables directory: {PUB_TABLES}")


✓ Setup complete!
Data directory: /home/ashrafs/projects/Conspiracy-Proj/Data
Tables directory: /home/ashrafs/projects/Conspiracy-Proj/publication/tables


## 1. Load Human Annotations


In [4]:
# Load Human Annotations
print("Loading human annotations...")

human_class_path = DATA / 'all_classifications.csv'
if not human_class_path.exists():
    raise FileNotFoundError(f"Human classification file not found: {human_class_path}")

df = pd.read_csv(human_class_path)
print(f"Loaded {len(df):,} annotation rows")

# Normalize booleans
BOOL_MAP = {'true': True, 'false': False, 'yes': True, 'no': False, 'y': True, 'n': False, '1': True, '0': False}
def parse_bool(s: pd.Series) -> pd.Series:
    v = s.astype(str).str.strip().str.lower().map(BOOL_MAP)
    unmapped = v.isna()
    if unmapped.any():
        num = pd.to_numeric(s[unmapped], errors='coerce')
        v.loc[unmapped & num.notna()] = (num[num.notna()] > 0).astype(bool)
    return v

# Parse conspiracy column
df['conspiracy_bool'] = parse_bool(df['conspiracy'])

# Get unique annotators
annotators = sorted(df['annotator_id'].unique())
print(f"\nFound {len(annotators)} unique annotators: {annotators}")
print(f"Total texts annotated: {df['text_id'].nunique():,} unique texts")


Loading human annotations...
Loaded 3,243 annotation rows

Found 13 unique annotators: [np.int64(4), np.int64(7), np.int64(17), np.int64(18), np.int64(60), np.int64(161), np.int64(165), np.int64(166), np.int64(172), np.int64(175), np.int64(178), np.int64(181), np.int64(182)]
Total texts annotated: 2,077 unique texts


## 2. Calculate Pairwise Cohen's Kappa


In [5]:
# Function to calculate Cohen's Kappa for a pair of annotators
def calculate_pairwise_kappa(annotator1_id, annotator2_id, df):
    """Calculate Cohen's Kappa for two annotators on texts they both annotated."""
    # Get annotations from both annotators
    ann1 = df[df['annotator_id'] == annotator1_id][['text_id', 'conspiracy_bool']].copy()
    ann2 = df[df['annotator_id'] == annotator2_id][['text_id', 'conspiracy_bool']].copy()
    
    # Merge on text_id to find texts annotated by both
    merged = ann1.merge(ann2, on='text_id', suffixes=('_1', '_2'), how='inner')
    
    if len(merged) == 0:
        return None, 0
    
    # Filter out rows where either annotator has NaN (only use texts with valid annotations from both)
    merged = merged.dropna(subset=['conspiracy_bool_1', 'conspiracy_bool_2'])
    
    if len(merged) == 0:
        return None, 0
    
    # Ensure boolean type (fill any remaining NaN as False, though shouldn't be any)
    merged['conspiracy_bool_1'] = merged['conspiracy_bool_1'].fillna(False).astype(bool)
    merged['conspiracy_bool_2'] = merged['conspiracy_bool_2'].fillna(False).astype(bool)
    
    # Convert boolean to int (0 = False/No, 1 = True/Yes)
    y1 = merged['conspiracy_bool_1'].astype(int).values
    y2 = merged['conspiracy_bool_2'].astype(int).values
    
    # Calculate Cohen's Kappa
    try:
        kappa = cohen_kappa_score(y1, y2, labels=[0, 1])
        if pd.isna(kappa):
            return None, len(merged)
    except Exception as e:
        return None, len(merged)
    
    return kappa, len(merged)

# Calculate pairwise Kappa for all annotator pairs
pairwise_results = []
annotator_pairs = list(combinations(annotators, 2))

print(f"Calculating pairwise Cohen's Kappa for {len(annotator_pairs)} annotator pairs...\n")

for ann1, ann2 in annotator_pairs:
    kappa, n_texts = calculate_pairwise_kappa(ann1, ann2, df)
    
    if kappa is not None:
        pairwise_results.append({
            'Annotator_1': ann1,
            'Annotator_2': ann2,
            'Kappa': kappa,
            'N_Texts': n_texts
        })
        print(f"  Pair ({ann1}, {ann2}): κ = {kappa:.4f} (n={n_texts})")
    else:
        print(f"  Pair ({ann1}, {ann2}): No overlapping texts or calculation error (n={n_texts})")

pairwise_df = pd.DataFrame(pairwise_results)
print(f"\n✓ Calculated Kappa for {len(pairwise_df)} pairs")


Calculating pairwise Cohen's Kappa for 78 annotator pairs...

  Pair (4, 7): κ = 0.5728 (n=98)
  Pair (4, 17): No overlapping texts or calculation error (n=0)
  Pair (4, 18): No overlapping texts or calculation error (n=0)
  Pair (4, 60): No overlapping texts or calculation error (n=0)
  Pair (4, 161): κ = 0.4646 (n=99)
  Pair (4, 165): No overlapping texts or calculation error (n=0)
  Pair (4, 166): No overlapping texts or calculation error (n=0)
  Pair (4, 172): No overlapping texts or calculation error (n=0)
  Pair (4, 175): No overlapping texts or calculation error (n=0)
  Pair (4, 178): No overlapping texts or calculation error (n=0)
  Pair (4, 181): No overlapping texts or calculation error (n=0)
  Pair (4, 182): No overlapping texts or calculation error (n=0)
  Pair (7, 17): κ = 0.4796 (n=100)
  Pair (7, 18): No overlapping texts or calculation error (n=0)
  Pair (7, 60): No overlapping texts or calculation error (n=0)
  Pair (7, 161): κ = 0.7167 (n=100)
  Pair (7, 165): No over

## 3. Create Pairwise Agreement Matrix


In [6]:
# Create a symmetric matrix of pairwise Kappa scores
if len(pairwise_df) > 0:
    # Initialize matrix with NaN
    kappa_matrix = pd.DataFrame(
        index=annotators,
        columns=annotators,
        dtype=float
    )
    
    # Fill diagonal (annotator with themselves) - should be 1.0
    np.fill_diagonal(kappa_matrix.values, 1.0)
    
    # Fill pairwise values (symmetric matrix)
    for _, row in pairwise_df.iterrows():
        ann1 = row['Annotator_1']
        ann2 = row['Annotator_2']
        kappa = row['Kappa']
        
        kappa_matrix.loc[ann1, ann2] = kappa
        kappa_matrix.loc[ann2, ann1] = kappa  # Make symmetric
    
    print("Pairwise Kappa Agreement Matrix:")
    print("=" * 80)
    display(kappa_matrix.round(4))
else:
    print("No pairwise results to display.")


Pairwise Kappa Agreement Matrix:


,4,7,17,18,60,161,165,166,172,175,178,181,182
4,1.0000,0.5728,NaN,NaN,NaN,0.4646,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.5728,1.0000,0.4796,NaN,NaN,0.7167,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,NaN,0.4796,1.0000,NaN,NaN,0.4877,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,1.0000,NaN,NaN,0.1828,NaN,NaN,NaN,0.0593,NaN,NaN
60,NaN,NaN,NaN,NaN,1.0000,NaN,NaN,NaN,NaN,0.4472,NaN,NaN,0.2398
161,0.4646,0.7167,0.4877,NaN,NaN,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
165,NaN,NaN,NaN,0.1828,NaN,NaN,1.0000,NaN,NaN,NaN,0.4993,NaN,NaN
166,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,NaN,NaN,NaN,0.4145,NaN
172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,0.6549,NaN,NaN,NaN
175,NaN,NaN,NaN,NaN,0.4472,NaN,NaN,NaN,0.6549,1.0000,NaN,NaN,0.1675


## 4. Summary Statistics


In [7]:
# Summary statistics
if len(pairwise_df) > 0:
    print("=" * 80)
    print("PAIRWISE COHEN'S KAPPA SUMMARY STATISTICS")
    print("=" * 80)
    
    mean_kappa = pairwise_df['Kappa'].mean()
    median_kappa = pairwise_df['Kappa'].median()
    std_kappa = pairwise_df['Kappa'].std()
    min_kappa = pairwise_df['Kappa'].min()
    max_kappa = pairwise_df['Kappa'].max()
    min_pair = pairwise_df.loc[pairwise_df['Kappa'].idxmin(), ['Annotator_1', 'Annotator_2']].values
    max_pair = pairwise_df.loc[pairwise_df['Kappa'].idxmax(), ['Annotator_1', 'Annotator_2']].values
    
    print(f"\nMean Kappa: {mean_kappa:.4f}")
    print(f"Median Kappa: {median_kappa:.4f}")
    print(f"Standard Deviation: {std_kappa:.4f}")
    print(f"Min Kappa: {min_kappa:.4f} (Pair: {min_pair[0]}, {min_pair[1]})")
    print(f"Max Kappa: {max_kappa:.4f} (Pair: {max_pair[0]}, {max_pair[1]})")
    
    # Interpretation based on Landis & Koch (1977)
    print(f"\nInterpretation (Landis & Koch, 1977):")
    print(f"  < 0.00: Poor agreement")
    print(f"  0.00-0.20: Slight agreement")
    print(f"  0.21-0.40: Fair agreement")
    print(f"  0.41-0.60: Moderate agreement")
    print(f"  0.61-0.80: Substantial agreement")
    print(f"  0.81-1.00: Almost perfect agreement")
    
    # Count by interpretation category
    categories = {
        'Poor (<0.00)': (pairwise_df['Kappa'] < 0.00).sum(),
        'Slight (0.00-0.20)': ((pairwise_df['Kappa'] >= 0.00) & (pairwise_df['Kappa'] <= 0.20)).sum(),
        'Fair (0.21-0.40)': ((pairwise_df['Kappa'] >= 0.21) & (pairwise_df['Kappa'] <= 0.40)).sum(),
        'Moderate (0.41-0.60)': ((pairwise_df['Kappa'] >= 0.41) & (pairwise_df['Kappa'] <= 0.60)).sum(),
        'Substantial (0.61-0.80)': ((pairwise_df['Kappa'] >= 0.61) & (pairwise_df['Kappa'] <= 0.80)).sum(),
        'Almost Perfect (0.81-1.00)': (pairwise_df['Kappa'] >= 0.81).sum(),
    }
    
    print(f"\nDistribution by Agreement Level:")
    for category, count in categories.items():
        pct = (count / len(pairwise_df)) * 100 if len(pairwise_df) > 0 else 0
        print(f"  {category}: {count} pairs ({pct:.1f}%)")
    
    print("=" * 80)
else:
    print("No pairwise results for summary statistics.")


PAIRWISE COHEN'S KAPPA SUMMARY STATISTICS

Mean Kappa: 0.4144
Median Kappa: 0.4646
Standard Deviation: 0.1970
Min Kappa: 0.0593 (Pair: 18.0, 178.0)
Max Kappa: 0.7167 (Pair: 7.0, 161.0)

Interpretation (Landis & Koch, 1977):
  < 0.00: Poor agreement
  0.00-0.20: Slight agreement
  0.21-0.40: Fair agreement
  0.41-0.60: Moderate agreement
  0.61-0.80: Substantial agreement
  0.81-1.00: Almost perfect agreement

Distribution by Agreement Level:
  Poor (<0.00): 0 pairs (0.0%)
  Slight (0.00-0.20): 3 pairs (23.1%)
  Fair (0.21-0.40): 1 pairs (7.7%)
  Moderate (0.41-0.60): 7 pairs (53.8%)
  Substantial (0.61-0.80): 2 pairs (15.4%)
  Almost Perfect (0.81-1.00): 0 pairs (0.0%)


## 5. Display Pairwise Results Table


In [10]:
# Display sorted pairwise results
if len(pairwise_df) > 0:
    # Sort by Kappa (descending)
    pairwise_sorted = pairwise_df.sort_values('Kappa', ascending=False).copy()
    
    print("=" * 80)
    print("PAIRWISE COHEN'S KAPPA AGREEMENT (Sorted by Kappa)")
    print("=" * 80)
    print(f"\nTask: Conspiracy Classification (Binary: Yes/No)")
    print(f"Total Pairs: {len(pairwise_sorted)}")
    print(f"\n{'='*80}\n")
    
    display(pairwise_sorted.round(4))
    
    print("\n" + "=" * 80)


PAIRWISE COHEN'S KAPPA AGREEMENT (Sorted by Kappa)

Task: Conspiracy Classification (Binary: Yes/No)
Total Pairs: 13




,Annotator_1,Annotator_2,Kappa,N_Texts
3,7,161,0.7167,100
11,172,175,0.6549,26
0,4,7,0.5728,98
9,165,178,0.4993,99
4,17,161,0.4877,100
2,7,17,0.4796,100
1,4,161,0.4646,99
7,60,175,0.4472,48
10,166,181,0.4145,100
8,60,182,0.2398,100


## 6. Export Results


In [ ]:
# Export as CSV
if len(pairwise_df) > 0:
    csv_path = PUB_TABLES / 'pairwise_kappa_agreement.csv'
    # Ensure annotator IDs are integers for CSV export
    export_df = pairwise_sorted[['Annotator_1', 'Annotator_2', 'Kappa', 'N_Texts']].copy()
    export_df['Annotator_1'] = export_df['Annotator_1'].astype(int)
    export_df['Annotator_2'] = export_df['Annotator_2'].astype(int)
    export_df.to_csv(
        csv_path, index=False, float_format='%.4f'
    )
    print(f"✓ Exported CSV: {csv_path}")
    
    # Export pairwise table as LaTeX
    latex_path = PUB_TABLES / 'pairwise_kappa_agreement.tex'
    
    latex_str = "\\begin{table*}[t]\n"
    latex_str += "\\centering\n"
    latex_str += "\\caption{Pairwise Cohen's Kappa Agreement for Conspiracy Classification}\n"
    latex_str += "\\label{tab:pairwise_kappa}\n"
    latex_str += "\\begin{tabular}{lcccc}\n"
    latex_str += "\\toprule\n"
    latex_str += "Annotator 1 & Annotator 2 & Kappa & N Texts \\\\\n"
    latex_str += "\\midrule\n"
    
    for _, row in pairwise_sorted.iterrows():
        # Convert annotator IDs to integers for clean display
        ann1 = int(row['Annotator_1'])
        ann2 = int(row['Annotator_2'])
        latex_str += f"{ann1} & {ann2} & {row['Kappa']:.4f} & {int(row['N_Texts'])} \\\\\n"
    
    latex_str += "\\bottomrule\n"
    latex_str += "\\end{tabular}\n"
    latex_str += "\\end{table*}\n"
    
    with open(latex_path, 'w') as f:
        f.write(latex_str)
    
    print(f"✓ Exported LaTeX table: {latex_path}")
    
    # Export summary statistics as text file
    summary_path = PUB_TABLES / 'pairwise_kappa_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("Pairwise Cohen's Kappa Agreement Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Mean Kappa: {mean_kappa:.4f}\n")
        f.write(f"Median Kappa: {median_kappa:.4f}\n")
        f.write(f"Standard Deviation: {std_kappa:.4f}\n")
        f.write(f"Min Kappa: {min_kappa:.4f}\n")
        f.write(f"Max Kappa: {max_kappa:.4f}\n")
        f.write(f"\nTotal Pairs: {len(pairwise_df)}\n")
    
    print(f"✓ Exported summary: {summary_path}")
    
    print("\n" + "=" * 80)
    print("Export Summary:")
    print(f"  - CSV: {csv_path}")
    print(f"  - LaTeX: {latex_path}")
    print(f"  - Summary: {summary_path}")
    print("=" * 80)
else:
    print("No pairwise results to export.")


✓ Exported CSV: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_agreement.csv
✓ Exported LaTeX table: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_agreement.tex
✓ Exported summary: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_summary.txt

Export Summary:
  - CSV: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_agreement.csv
  - LaTeX: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_agreement.tex
  - Summary: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/pairwise_kappa_summary.txt
